# GNSS Single Point Positioning with Factor Graphs

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/SinglePointPositioningExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview

The `PseudorangeFactor` family provides factors for incorporating GNSS pseudorange measurements into a GTSAM factor graph. Pseudorange factors model the distance between a satellite and the GNSS receiver along with their associated errors (atmospheric, multipath, clock bias, etc...). This more-tightly couples GNSS positioning within the general probabilistic graphical estimation framework compared to loosely-coupled `GPSFactor`s. In other words, `PseudorangeFactor` integrates GNSS measurements in a "raw" form so the factor graph has the opportunity to correct GNSS measurements as part of the overall optimization process. Tighly-coupled GNSS filters also enable partial position observability even if the requisite minimum 4 satellites for an independent position calculation are not met.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

In [1]:
%pip install --quiet pyrtklib numpy


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Usage Example

In [2]:
import numpy as np
import pyrtklib as rtklib

# TODO: Remove this before merging!!!
import sys
sys.path.insert(0, "/Users/sammy/gtsam-gnss/_build/python/gtsam")
import gtsam
from gtsam.symbol_shorthand import X, B


obs = rtklib.obs_t()
nav = rtklib.nav_t()
sta = rtklib.sta_t()
load_status = rtklib.readrnx("/Users/sammy/gtsam-gnss/examples/Data/brdc0180.26n", 1, "", obs, nav, sta)
assert load_status == 1, "Loading broadcast ephemeris RINEX"
load_status = rtklib.readrnx("/Users/sammy/gtsam-gnss/examples/Data/zoa10180.26o", 1, "", obs, nav, sta)
assert load_status == 1, "Loading receiver observation RINEX"

print(f"Found {obs.n} observations")
print(f"Found {nav.n} navigation messages")

Found 941392 observations
Found 472 navigation messages


In [32]:
n = 11                         # number of observations to process.
rs = rtklib.Arr1Ddouble(6*n)   # Satellite positions and velocities:
                               # rs [(0:2)+i*6] = obs[i] sat position {x,y,z} (m)
                               # rs [(3:5)+i*6] = obs[i] sat velocity {vx,vy,vz} (m/s)
dts = rtklib.Arr1Ddouble(2*n)  # Satellite clock drift:
                               # dts[(0:1)+i*2] = obs[i] sat clock {bias,drift} (s|s/s)
var = rtklib.Arr1Ddouble(1*n)  # Satellite position and clock variances:
                               # var[i] = obs[i] sat position and clock error variance (m^2)
svh = rtklib.Arr1Dint(1*n)     # Satellite health flag (-1:correction not available):
                               # svh[i] = obs[i] sat health flag

teph = obs.data[0].time

# Compute satellite positions:
rtklib.satposs(
    teph,              # gtime_t teph     I   time to select ephemeris (gpst)
    obs.data.ptr,      # obsd_t *obs      I   observation data
    n,                 # int    n         I   number of observation data
    nav,               # nav_t  *nav      I   navigation data
    0,                 # int    ephopt    I   ephemeris option (EPHOPT_???) (EPHOPT_BRDC == 0)
    rs,                # double *rs       O   satellite positions and velocities (ecef)
    dts,               # double *dts      O   satellite clocks
    var,               # double *var      O   sat position and clock error variances (m^2)
    svh                # int    *svh      O   sat health flag (-1:correction not available)
)

In [37]:
# Prepare factor graph:
cb_key = B(0)  # Receiver clock bias variable key.
pos_key = X(0) # Receiver position variable key.
nm = gtsam.noiseModel.Diagonal.Sigmas(np.array([1.0]))
graph = gtsam.NonlinearFactorGraph()

# Add a PseudorangeFactor for each observation:
sats_used = set()
for i in range(n):
    obsd = obs.data[i]
    sats_used.add(obsd.sat)
    sat_bias = dts[2*i+0]
    sat_drift = dts[2*i+1]
    sat_pos = np.array([rs[6*i+0], rs[6*i+1], rs[6*i+2]])

    # Identify pseudorange code CODE_L1C:
    for j, code in enumerate(obsd.code):
        if code == rtklib.CODE_L1C:
            pseudorange = obsd.P[j]
            break

    graph.add(
        gtsam.PseudorangeFactor(pos_key, cb_key, pseudorange, sat_pos, sat_bias, nm)
    )

    # Comment to print debug lines below:
    continue
    
    print(f"=== Satellite {obsd.sat} ===")
    print(f"Receiver pseudorange: {pseudorange} meters")
    print(f"Observation time (unixtime): {rtklib.time_str(obsd.time, 3)} ({obsd.time.time})")
    print(f"Satellite position (ECEF): {1e-3*sat_pos} km")
    # print(f"Satellite velocity (ECEF): {rs[6*i+3]}, {rs[6*i+4]}, {rs[6*i+5]} m/s")
    print(f"Satellite clock drift: bias {sat_bias} seconds, drift {sat_drift} seconds-per-second")
    # print(f"Satellite variance: {var[i]}")
    # print(f"Satellite health: {svh[i]}")
    print()

# Solve the system:
initial_values = gtsam.Values()
initial_values.insert(cb_key, 0.0)
initial_values.insert(pos_key, np.zeros(3))
result = gtsam.LevenbergMarquardtOptimizer(graph, initial_values).optimize()

In [39]:
# Process the results:
clock_bias = result.atDouble(cb_key)
receiver_position = result.atVector(pos_key)
print(f"Final clock bias {clock_bias} seconds and receiver position {1e-3*receiver_position} (ECEF km)")
print(f"Using the following satellites: {sats_used}")

Final clock bias 6.20908953458917e-07 seconds and receiver position [-2684.43802288 -4293.39350686  3865.378281  ] (ECEF km)
Using the following satellites: {5, 6, 11, 12, 19, 21, 22, 24, 25, 28, 29}


## Source
- [PseudorangeFactor.h](https://github.com/borglab/gtsam/blob/develop/gtsam/navigation/PseudorangeFactor.h)
- [PseudorangeFactor.cpp](https://github.com/borglab/gtsam/blob/develop/gtsam/navigation/PseudorangeFactor.cpp)